In [1]:
import pandas as pd

train = pd.read_csv("../data/processed/chicago_crimes_2015_2024_cleaned.csv")
test = pd.read_csv("../data/processed/chicago_crimes_2025_cleaned.csv")

print(train.shape)
print(test.shape)

(2477268, 13)
(236595, 13)


In [2]:
# Keep only the variables required for further analysis and prediction
columns_to_keep = [
    "date",
    "primary_type",
    "location_description",
    "arrest",
    "beat",
    "district",
    "ward",
    "community_area",
    "latitude",
    "longitude"
]

train = train[columns_to_keep].copy()
test = test[columns_to_keep].copy()

# Display the column names after filtering
print("Filtered train columns:", train.columns.tolist())
print("Filtered test columns:", test.columns.tolist())

Filtered train columns: ['date', 'primary_type', 'location_description', 'arrest', 'beat', 'district', 'ward', 'community_area', 'latitude', 'longitude']
Filtered test columns: ['date', 'primary_type', 'location_description', 'arrest', 'beat', 'district', 'ward', 'community_area', 'latitude', 'longitude']


In [3]:
# Remove rows with missing latitude or longitude because coordinates are essential for spatial analysis
train = train.dropna(subset=["latitude", "longitude"])
test = test.dropna(subset=["latitude", "longitude"])

# Check missing values in other important variables
print("Missing values in train dataset:")
print(train[["location_description", "community_area"]].isnull().sum())

print("\nMissing values in test dataset:")
print(test[["location_description", "community_area"]].isnull().sum())

Missing values in train dataset:
location_description    8247
community_area           174
dtype: int64

Missing values in test dataset:
location_description    1088
community_area             3
dtype: int64


In [4]:
# Fill missing location_description values with 'UNKNOWN'
train["location_description"] = train["location_description"].fillna("UNKNOWN")
test["location_description"] = test["location_description"].fillna("UNKNOWN")

# Fill missing community_area values with the mode of each dataset
community_mode = train["community_area"].mode()[0]
train["community_area"] = train["community_area"].fillna(community_mode)
test["community_area"] = test["community_area"].fillna(community_mode)

# Verify that missing values have been handled
print("Remaining missing values in train dataset:")
print(train[["location_description", "community_area"]].isnull().sum())

print("\nRemaining missing values in test dataset:")
print(test[["location_description", "community_area"]].isnull().sum())

Remaining missing values in train dataset:
location_description    0
community_area          0
dtype: int64

Remaining missing values in test dataset:
location_description    0
community_area          0
dtype: int64


In [5]:
# Remove duplicate crime records based on date and beat
train = train.drop_duplicates(subset=["date", "beat"])
test = test.drop_duplicates(subset=["date", "beat"])

# Display dataset shapes after duplicate removal
print("Train shape after duplicate removal:", train.shape)
print("Test shape after duplicate removal:", test.shape)

Train shape after duplicate removal: (2428229, 10)
Test shape after duplicate removal: (230460, 10)


In [6]:
# Convert the date column to datetime format
train["date"] = pd.to_datetime(train["date"], errors="coerce")
test["date"] = pd.to_datetime(test["date"], errors="coerce")

# Remove rows where date conversion failed
train = train.dropna(subset=["date"])
test = test.dropna(subset=["date"])

# Convert selected columns to categorical type for better memory efficiency and model preparation
categorical_cols = ["primary_type", "location_description"]

for col in categorical_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

#transform 'arrest' to int form
train["arrest"] = train["arrest"].astype(int)
test["arrest"] = test["arrest"].astype(int)

# Check the data types after conversion
print(train.dtypes)

date                    datetime64[ns]
primary_type                  category
location_description          category
arrest                           int64
beat                             int64
district                       float64
ward                           float64
community_area                 float64
latitude                       float64
longitude                      float64
dtype: object


In [7]:
# Define the approximate geographic boundary of Chicago
min_lat, max_lat = 41.64, 42.02
min_lon, max_lon = -87.94, -87.52

# Remove rows with invalid geographic coordinates
train = train[
    train["latitude"].between(min_lat, max_lat) &
    train["longitude"].between(min_lon, max_lon)
]

test = test[
    test["latitude"].between(min_lat, max_lat) &
    test["longitude"].between(min_lon, max_lon)
]

# Display final dataset shapes after coordinate validation
print("Final train shape:", train.shape)
print("Final test shape:", test.shape)

Final train shape: (2424241, 10)
Final test shape: (230163, 10)


In [8]:
#feature engineering: Group detailed location descriptions into broader categories
# This helps reduce sparsity and improve model performance

# Define mapping rules for location categories
location_map = {
    
    # Public outdoor spaces
    "STREET": "PUBLIC_OUTDOOR",
    "SIDEWALK": "PUBLIC_OUTDOOR",
    "ALLEY": "PUBLIC_OUTDOOR",
    "PARK": "PUBLIC_OUTDOOR",
    "HIGHWAY": "PUBLIC_OUTDOOR",
    
    # Residential areas
    "APARTMENT": "RESIDENTIAL",
    "RESIDENCE": "RESIDENTIAL",
    "HOUSE": "RESIDENTIAL",
    
    # Commercial locations
    "RESTAURANT": "COMMERCIAL",
    "BAR": "COMMERCIAL",
    "GAS STATION": "COMMERCIAL",
    "STORE": "COMMERCIAL",
    
    # Institutional locations
    "SCHOOL": "INSTITUTION",
    "HOSPITAL": "INSTITUTION"
}

# Apply the mapping to create a new feature
train["location_category"] = train["location_description"].map(location_map)
test["location_category"] = test["location_description"].map(location_map)

# Assign "OTHER" to locations that are not explicitly mapped
train["location_category"] = train["location_category"].fillna("OTHER")
test["location_category"] = test["location_category"].fillna("OTHER")

# Convert the new feature to categorical type
train["location_category"] = train["location_category"].astype("category")
test["location_category"] = test["location_category"].astype("category")

# Check the distribution of the new location categories
print("Train location_category distribution:")
print(train["location_category"].value_counts())

print("\nTest location_category distribution:")
print(test["location_category"].value_counts())

Train location_category distribution:
location_category
PUBLIC_OUTDOOR    825125
OTHER             760308
RESIDENTIAL       753658
COMMERCIAL         85138
INSTITUTION           12
Name: count, dtype: int64

Test location_category distribution:
location_category
PUBLIC_OUTDOOR    77225
RESIDENTIAL       72642
OTHER             72386
COMMERCIAL         7909
INSTITUTION           1
Name: count, dtype: int64


In [9]:
# One-Hot Encode the location_category feature
# This converts categorical values into binary indicator variables for machine learning models

train = pd.get_dummies(train, columns=["location_category"], prefix="loc")
test = pd.get_dummies(test, columns=["location_category"], prefix="loc")

# Ensure train and test have the same columns after encoding
train, test = train.align(test, join="left", axis=1, fill_value=0)

# Check the new columns
print("Train columns after one-hot encoding:")
print(train.columns)

print("\nTest columns after one-hot encoding:")
print(test.columns)

Train columns after one-hot encoding:
Index(['date', 'primary_type', 'location_description', 'arrest', 'beat',
       'district', 'ward', 'community_area', 'latitude', 'longitude',
       'loc_COMMERCIAL', 'loc_INSTITUTION', 'loc_OTHER', 'loc_PUBLIC_OUTDOOR',
       'loc_RESIDENTIAL'],
      dtype='object')

Test columns after one-hot encoding:
Index(['date', 'primary_type', 'location_description', 'arrest', 'beat',
       'district', 'ward', 'community_area', 'latitude', 'longitude',
       'loc_COMMERCIAL', 'loc_INSTITUTION', 'loc_OTHER', 'loc_PUBLIC_OUTDOOR',
       'loc_RESIDENTIAL'],
      dtype='object')


In [10]:
# Convert boolean columns (True/False) to numeric (1/0)
train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

bool_cols = train.select_dtypes(include="bool").columns

train[bool_cols] = train[bool_cols].astype(int)
test[bool_cols] = test[bool_cols].astype(int)

print("Converted boolean columns to 0/1:")
print(bool_cols)

Converted boolean columns to 0/1:
Index(['loc_COMMERCIAL', 'loc_INSTITUTION', 'loc_OTHER', 'loc_PUBLIC_OUTDOOR',
       'loc_RESIDENTIAL'],
      dtype='object')


In [11]:
# Preview the processed datasets
print("Processed train dataset preview:")
display(train.head())

print("Processed test dataset preview:")
display(test.head())

Processed train dataset preview:


,date,primary_type,location_description,arrest,beat,district,ward,community_area,latitude,longitude,loc_COMMERCIAL,loc_INSTITUTION,loc_OTHER,loc_PUBLIC_OUTDOOR,loc_RESIDENTIAL
0,2015-03-19 16:47:00,BATTERY,STREET,1,2535,25.0,26.0,23.0,41.906354,-87.718554,0,0,0,1,0
1,2015-03-22 02:34:00,BATTERY,HOTEL/MOTEL,1,833,8.0,13.0,65.0,41.759284,-87.741709,0,0,1,0,0
2,2015-03-26 15:45:00,THEFT,SMALL RETAIL STORE,1,2422,24.0,49.0,1.0,42.019399,-87.675049,0,0,1,0,0
3,2015-03-31 12:00:00,BURGLARY,RESIDENCE,0,2211,22.0,19.0,74.0,41.683936,-87.721847,0,0,0,0,1
4,2015-03-31 20:41:00,BATTERY,APARTMENT,1,935,9.0,3.0,61.0,41.795248,-87.643360,0,0,0,0,1


Processed test dataset preview:


,date,primary_type,location_description,arrest,beat,district,ward,community_area,latitude,longitude,loc_COMMERCIAL,loc_INSTITUTION,loc_OTHER,loc_PUBLIC_OUTDOOR,loc_RESIDENTIAL
0,2025-01-01 03:57:00,HOMICIDE,YARD,0,634,6,9.0,49.0,41.722435,-87.632169,0,0,1,0,0
1,2025-01-01 06:10:00,HOMICIDE,APARTMENT,1,1024,10,22.0,30.0,41.850530,-87.714967,0,0,0,0,1
2,2025-01-01 14:06:00,HOMICIDE,TAVERN,0,612,6,17.0,71.0,41.756770,-87.653906,0,0,1,0,0
3,2025-01-02 01:18:00,HOMICIDE,HOUSE,1,1022,10,24.0,29.0,41.865616,-87.706862,0,0,0,0,1
4,2025-01-04 18:34:00,HOMICIDE,STREET,0,631,6,8.0,44.0,41.747660,-87.602223,0,0,0,1,0


In [12]:
# feature engineering
# Random Forest Feature Importance Analysis


from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

# Define target variable
y_train = train["arrest"]

# Define feature set (exclude non-feature columns)
X_train = train.drop(columns=["arrest", "date", "primary_type", "location_description"])

# Train Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Extract feature importance scores
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
})

# Sort features by importance
feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

# Display top important features
print("Random Forest Feature Importance:")
display(feature_importance)

Random Forest Feature Importance:


,feature,importance
4,latitude,0.473604
5,longitude,0.402574
0,beat,0.032241
2,ward,0.021337
10,loc_RESIDENTIAL,0.020944
3,community_area,0.018063
9,loc_PUBLIC_OUTDOOR,0.012339
1,district,0.011858
8,loc_OTHER,0.005241
6,loc_COMMERCIAL,0.001790


In [13]:
import os

os.makedirs("../data/furtherprocessed", exist_ok=True)

# Save the processed datasets for downstream analysis and modeling
train.to_csv("../data/furtherprocessed/chicago_crimes_2015_2024_processed.csv", index=False)
test.to_csv("../data/furtherprocessed/chicago_crimes_2025_processed.csv", index=False)

print("Processed files have been saved successfully.")

Processed files have been saved successfully.
